In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, explained_variance_score
from sklearn.model_selection import GridSearchCV
from matplotlib import pyplot as plt

In [ ]:
rounding = 3

In [11]:
def print_feature_importances(cols, importances):
    idx = np.argsort(importances)[::-1]
    print(list(zip(np.array(cols)[idx], np.array(importances)[idx])))

def feature_importance_dict(cols, importances):
    dict = {}
    for i in range(len(cols)):
        col = cols[i]
        dict[col] = np.round(importances[i], rounding)
    return dict

In [30]:
df = pd.read_csv('../processed_data/processed_data.csv')
basin_dict = {"AL": 0.0, "CP": 2.0, "EP": 1.0}
for i in range(len(df)):
    df.at[i, "Basin"] = basin_dict[df.iloc[i]["Basin"]]
df["Basin"]

0       0.0
1       0.0
2       0.0
3       0.0
4       0.0
       ... 
7381    1.0
7382    1.0
7383    1.0
7384    1.0
7385    1.0
Name: Basin, Length: 7386, dtype: object

In [26]:
cols = ["logVMAX12", "MSLP12", "POT12", "VGRAD0-6", "VGRAD6-12", "ONI", "Basin"]
X, y = df[cols].to_numpy(), df["logVMAX36"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=4)

In [27]:
dtr = DecisionTreeRegressor()
# Grid search
params = {"criterion": ["squared_error", "friedman_mse", "absolute_error", "poisson"],
          "splitter": ["best"],
          "max_depth": [None, 3, 5, 10],
          "min_samples_split": [2, 3, 5, 8],
          "max_features": [None]}
dtr_gs = GridSearchCV(dtr, params, scoring="explained_variance", n_jobs=-1)

dtr_gs.fit(X_train, y_train)

print("Best params:", dtr_gs.best_params_)
print("Best score:", dtr_gs.best_score_)

Best params: {'criterion': 'squared_error', 'max_depth': None, 'max_features': None, 'min_samples_split': 2, 'splitter': 'best'}
Best score: 0.9804666192887626


In [35]:
dtr = DecisionTreeRegressor(**dtr_gs.best_params_)
fit_dtr = dtr.fit(X_train, y_train)
tree_importances_df = pd.DataFrame({"Importance": feature_importance_dict(cols, fit_dtr.feature_importances_)}).T
print(tree_importances_df.to_latex(caption="Feature Importances of sklearn Decision Tree Regressor with Basin Indicator", label="tab:tree_importances"))
tree_importances_df

\begin{table}
\caption{Feature Importances of sklearn Decision Tree Regressor with Basin Indicator}
\label{tab:tree_importances}
\begin{tabular}{lrrrrrrrr}
\toprule
 & logVMAX12 & MSLP12 & POT12 & VGRAD0-6 & VGRAD6-12 & ONI & LAT & LON \\
\midrule
Importance & 0.187000 & 0.076000 & 0.144000 & 0.020000 & 0.049000 & 0.108000 & 0.203000 & 0.215000 \\
\bottomrule
\end{tabular}
\end{table}



,logVMAX12,MSLP12,POT12,VGRAD0-6,VGRAD6-12,ONI,LAT,LON
Importance,0.187,0.076,0.144,0.02,0.049,0.108,0.203,0.215


In [36]:
y_pred = fit_dtr.predict(X_test)
y_train_pred = fit_dtr.predict(X_train)
out_of_sample = {"MSE": mean_squared_error(y_test, y_pred), "Explained Variance": explained_variance_score(y_test, y_pred)}
in_sample = {"MSE": mean_squared_error(y_train, y_train_pred), "Explained Variance": explained_variance_score(y_train, y_train_pred)}
tree_results = {"In Sample": in_sample, "Out of Sample": out_of_sample}
tree_results_df = pd.DataFrame(tree_results).T
print(tree_results_df.to_latex(caption="Scoring for sklearn sklearn Decision Tree Regressor with Basin Indicator", label="tab:tree_scoring"))
tree_results_df

\begin{table}
\caption{Scoring for sklearn sklearn Decision Tree Regressor with Basin Indicator}
\label{tab:tree_scoring}
\begin{tabular}{lrr}
\toprule
 & MSE & Explained Variance \\
\midrule
In Sample & 0.000001 & 0.999990 \\
Out of Sample & 0.001560 & 0.980684 \\
\bottomrule
\end{tabular}
\end{table}



,MSE,Explained Variance
In Sample,7.687907e-07,0.999990
Out of Sample,1.560219e-03,0.980684


In [ ]:
cols = ["logVMAX12", "MSLP12", "POT12", "VGRAD0-6", "VGRAD6-12", "ONI", "LAT", "LON"]
X, y = df[cols].to_numpy(), df["logVMAX36"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=4)

dtr = DecisionTreeRegressor()
# Grid search
params = {"criterion": ["squared_error", "friedman_mse", "absolute_error", "poisson"],
          "splitter": ["best"],
          "max_depth": [None, 3, 5, 10],
          "min_samples_split": [2, 3, 5, 8],
          "max_features": [None]}
dtr_gs = GridSearchCV(dtr, params, scoring="explained_variance", n_jobs=-1)

dtr_gs.fit(X_train, y_train)

print("Best params:", dtr_gs.best_params_)
print("Best score:", dtr_gs.best_score_)

dtr = DecisionTreeRegressor(**dtr_gs.best_params_)
fit_dtr = dtr.fit(X_train, y_train)
tree_importances_df = pd.DataFrame({"Importance": feature_importance_dict(cols, fit_dtr.feature_importances_)}).T
print(tree_importances_df.to_latex(caption="Feature Importances of sklearn Decision Tree Regressor with Latitude and Longitude", label="tab:latlon_tree_importances"))

y_pred = fit_dtr.predict(X_test)
y_train_pred = fit_dtr.predict(X_train)
out_of_sample = {"MSE": mean_squared_error(y_test, y_pred), "Explained Variance": explained_variance_score(y_test, y_pred)}
in_sample = {"MSE": mean_squared_error(y_train, y_train_pred), "Explained Variance": explained_variance_score(y_train, y_train_pred)}
tree_results = {"In Sample": in_sample, "Out of Sample": out_of_sample}
tree_results_df = pd.DataFrame(tree_results).T
print(tree_results_df.to_latex(caption="Scoring for sklearn sklearn Decision Tree Regressor with Latitude and Longitude", label="tab:latlon_tree_scoring"))

Best params: {'criterion': 'absolute_error', 'max_depth': None, 'max_features': None, 'min_samples_split': 3, 'splitter': 'best'}
Best score: 0.9762528606732059
\begin{table}
\caption{Feature Importances of sklearn Decision Tree Regressor}
\label{tab:tree_importances}
\begin{tabular}{lrrrrrrrr}
\toprule
 & logVMAX12 & MSLP12 & POT12 & VGRAD0-6 & VGRAD6-12 & ONI & LAT & LON \\
\midrule
Importance & 0.182000 & 0.066000 & 0.160000 & 0.015000 & 0.049000 & 0.122000 & 0.195000 & 0.211000 \\
\bottomrule
\end{tabular}
\end{table}

\begin{table}
\caption{Scoring for sklearn sklearn Decision Tree Regressor}
\label{tab:tree_scoring}
\begin{tabular}{lrr}
\toprule
 & MSE & Explained Variance \\
\midrule
In Sample & 0.000000 & 1.000000 \\
Out of Sample & 0.001496 & 0.981482 \\
\bottomrule
\end{tabular}
\end{table}



In [33]:
tree_importances_df

,logVMAX12,MSLP12,POT12,VGRAD0-6,VGRAD6-12,ONI,LAT,LON
Importance,0.182,0.066,0.16,0.015,0.049,0.122,0.195,0.211


In [34]:
tree_results_df

,MSE,Explained Variance
In Sample,0.000000,1.000000
Out of Sample,0.001496,0.981482
